In [8]:
# Cell 1: Environment Setup

import os
import sys
import boto3
import json
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

# Clear AWS_PROFILE
if 'AWS_PROFILE' in os.environ:
    del os.environ['AWS_PROFILE']

# Configure environment
os.environ['USE_AWS'] = 'true'
os.environ['AWS_REGION'] = 'us-east-1'


os.environ['OPENSEARCH_ENDPOINT'] = 'https://search-ms-sos-legal-vector-awmjbt2db4a4gn4jhml5zpurm4.us-east-1.es.amazonaws.com'
os.environ['OPENSEARCH_INDEX'] = 'ms-legal-abstracts'
os.environ['BEDROCK_CLAUDE_MODEL'] = 'mistral.mistral-7b-instruct-v0:2'
os.environ['BEDROCK_EMBEDDING_MODEL'] = 'amazon.titan-embed-text-v1'

console = Console()
console.print("[bold green]✅ Environment configured![/bold green]")
console.print(f"[cyan]OpenSearch: {os.environ['OPENSEARCH_ENDPOINT']}[/cyan]")
console.print(f"[cyan]Model: {os.environ['BEDROCK_CLAUDE_MODEL']}[/cyan]")

✅ Environment configured!

OpenSearch: https://search-ms-sos-legal-vector-awmjbt2db4a4gn4jhml5zpurm4.us-east-1.es.amazonaws.com

Model: mistral.mistral-7b-instruct-v0:2

In [9]:
# Cell 2: Import Modules

import importlib
import sys

# Reload modules to pick up latest changes
for module in ['config', 'models', 'vector_store_opensearch', 'compression_agent_bedrock']:
    if module in sys.modules:
        importlib.reload(sys.modules[module])

from config import config
from vector_store_opensearch import OpenSearchVectorStore
from compression_agent_bedrock import BedrockCompressionAgent

console.print("[bold green]✅ Modules loaded![/bold green]")

✅ Modules loaded!

In [10]:
# Cell 3: Verify Index Contents

console.print("[bold blue]📊 OpenSearch Index Statistics[/bold blue]\n")

# Initialize vector store
vector_store = OpenSearchVectorStore()

# Get stats
try:
    stats = vector_store.get_stats()
    
    table = Table(title="Index Statistics")
    table.add_column("Metric", style="cyan")
    table.add_column("Value", style="green")
    
    table.add_row("Total Abstracts", str(stats.get('total_abstracts', 0)))
    table.add_row("Unique Documents", str(stats.get('unique_documents', 0)))
    table.add_row("Index Name", stats.get('index_name', 'N/A'))
    
    console.print(table)
    
    if stats.get('documents'):
        console.print("\n[bold cyan]📄 Indexed Documents:[/bold cyan]")
        for doc in stats['documents'][:10]:  # Show first 10
            console.print(f"  • {doc}")
    
    console.print(f"\n[bold green]✅ Found {stats.get('total_abstracts', 0)} abstracts from {stats.get('unique_documents', 0)} documents![/bold green]")
    
except Exception as e:
    console.print(f"[red]❌ Error: {e}[/red]")

📊 OpenSearch Index Statistics

            Index Statistics             
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Metric           ┃ Value              ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ Total Abstracts  │ 605                │
│ Unique Documents │ 11                 │
│ Index Name       │ ms-legal-abstracts │
└──────────────────┴────────────────────┘

📄 Indexed Documents:

• 00000506c.pdf

• 00000263c.pdf

...

• 00000228c.pdf

• 00000483c.pdf

• 00000586c.pdf

• 00000140c.pdf

✅ Found 605 abstracts from 11 documents!

In [4]:
# Cell 4: Test Search/Retrieval

console.print("[bold blue]🔍 Testing Retrieval[/bold blue]\n")

# Test queries
test_queries = [
    "What are the legal requirements?",
    "Tell me about compliance",
    "What penalties are mentioned?",
    "Explain the regulations",
]

for i, query in enumerate(test_queries, 1):
    console.print(f"[bold cyan]Query {i}:[/bold cyan] {query}")
    
    try:
        results = vector_store.search(query, top_k=3)
        
        if not results:
            console.print("[yellow]  ⚠️  No results[/yellow]\n")
            continue
        
        console.print(f"[green]  ✅ Found {len(results)} results[/green]")
        
        # Show top result
        top = results[0]
        console.print(f"[dim]  Document: {top.abstract.source_document}[/dim]")
        console.print(f"[dim]  Pages: {top.abstract.page_numbers}[/dim]")
        console.print(f"[dim]  Score: {top.similarity_score:.3f}[/dim]")
        console.print(f"[dim]  Abstract: {top.abstract.abstract_text[:200]}...[/dim]")
        console.print()
        
    except Exception as e:
        console.print(f"[red]  ❌ Error: {e}[/red]\n")

console.print("[bold green]✅ Retrieval working![/bold green]")

🔍 Testing Retrieval

Query 1: What are the legal requirements?

  ✅ Found 3 results

  Document: tmp221d6xvh.pdf

  Pages: [29, 30]

...

  Document: tmp221d6xvh.pdf

  Pages: [147, 148]

  Score: 0.735

  Abstract: This document outlines the Compliance Tariff 44, which includes provisions for electronic filing, form 
submission, opposition, temporary protective orders, time computation, trade secrets, transcripts...

✅ Retrieval working!

In [5]:
# Cell 5: Complete RAG Pipeline Test

console.print("[bold blue]🤖 Testing Complete RAG Pipeline[/bold blue]\n")

# Initialize Bedrock client for response generation
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

def query_rag(question: str, top_k: int = 3):
    """Complete RAG query"""
    
    # 1. Retrieve
    console.print(f"[dim]  🔍 Searching...[/dim]")
    results = vector_store.search(question, top_k=top_k)
    
    if not results:
        return "No relevant documents found.", []
    
    # 2. Build context
    context_parts = []
    citations = []
    
    for i, result in enumerate(results, 1):
        abstract = result.abstract
        context_parts.append(
            f"[Document {i}: {abstract.source_document}, Pages {abstract.page_numbers}]\n"
            f"Abstract: {abstract.abstract_text}\n"
            f"Core Rule: {abstract.core_rule or 'N/A'}\n"
        )
        citations.append({
            'document': abstract.source_document,
            'pages': abstract.page_numbers,
            'score': result.similarity_score
        })
    
    context = "\n".join(context_parts)
    
    # 3. Generate response
    console.print(f"[dim]  🤔 Generating answer...[/dim]")
    
    prompt = f"""<s>[INST] You are a legal research assistant. 

Based on these legal documents, answer the question accurately and cite sources.

CONTEXT:
{context}

QUESTION: {question}

Provide a clear answer based on the context above. Always mention which document you're referencing. [/INST]"""
    
    response = bedrock.invoke_model(
        modelId=config.aws.bedrock_claude_model,
        body=json.dumps({
            "prompt": prompt,
            "max_tokens": 1024,
            "temperature": 0.1,
            "top_p": 0.9
        })
    )
    
    result = json.loads(response['body'].read())
    answer = result['outputs'][0]['text'].strip()
    
    return answer, citations

# Test questions
test_questions = [
    "What are the main legal requirements mentioned in these documents?",
    "Are there any penalties or sanctions discussed?",
    "What compliance obligations are described?",
]

for question in test_questions:
    console.print(f"\n[bold cyan]❓ Question:[/bold cyan] {question}\n")
    
    try:
        answer, citations = query_rag(question)
        
        # Display answer
        console.print(Panel(
            f"{answer}",
            title="[bold green]🤖 Answer[/bold green]",
            border_style="green"
        ))
        
        # Display sources
        if citations:
            console.print("\n[bold cyan]📚 Sources:[/bold cyan]")
            for i, cite in enumerate(citations, 1):
                console.print(
                    f"  {i}. {cite['document']} "
                    f"(Pages: {cite['pages']}, "
                    f"Relevance: {cite['score']:.2f})"
                )
        
        console.print("\n" + "─" * 70)
        
    except Exception as e:
        console.print(f"[red]❌ Error: {e}[/red]\n")

console.print("\n[bold green]✅ Complete RAG pipeline working![/bold green]")

🤖 Testing Complete RAG Pipeline

❓ Question: What are the main legal requirements mentioned in these documents?

  🔍 Searching...

  🤔 Generating answer...

╭─────────────────────────────────────────────────── 🤖 Answer ───────────────────────────────────────────────────╮
│ The main legal requirements mentioned in the documents are as follows:                                          │
│                                                                                                                 │
│ 1. Compliance Tariff 44 (Document 1):                                                                           │
│    - Electronic filing and form submission                                                                      │
│    - Procedures for opposition                                                                                  │
│    - Temporary protective orders                                                                                │
│    - Time computation                                                                                           │
│    - Trade secrets protection                                                                                   │
│    - Transcripts                                                                                                │
│    - User's guide                                                                                               │
│    - Waiver                                                                                                     │
│    - Adherence to specified procedures                                                                          │
│    - Relevant statutes: Miss. Code Ann. § 28 (time computation), Miss. Code Ann. § 20 (trade secrets), and      │
│ Miss. Code Ann. § 29, § 60 (transcripts)                                                                        │
│                                                                                                                 │
│ 2. Commission's Procedures (Document 2):                                                                        │
│    - Requesting documents (informal and formal)                                                                 │
│    - Access to public computers                                                                                 │
│    - Operating the Commission's offices                                                                         │
│                                                                                                                 │
│ 3. Mississippi Rules of Appellate Procedure, Rule 46(b) (Document 3):                                           │
│    - Signing of all documents, except for prefiled testimony                                                    │
│    - Provision of contact information                                                                           │
│    - Certification by attorneys that they have read and believe the document is valid                           │
│    - Foreign attorneys must certify admission in accordance with Rule 46(b)                                     │
│                                                                                                                 │
│ These requirements are essential for adhering to the procedures outlined in the respective documents.           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

1. tmp221d6xvh.pdf (Pages: [147, 148], Relevance: 0.79)

2. tmp221d6xvh.pdf (Pages: [5, 6], Relevance: 0.75)

3. 00000047c.pdf (Pages: [5, 6], Relevance: 0.74)

──────────────────────────────────────────────────────────────────────

✅ Complete RAG pipeline working!

In [6]:
# Cell 6: Interactive Chat in Notebook

console.print(Panel.fit(
    "[bold blue]🤖 CLaRa Interactive Chat[/bold blue]\n"
    "[cyan]Ask questions about your legal documents[/cyan]",
    border_style="blue"
))

# Sample questions
console.print("\n[bold]Try these questions:[/bold]")
sample_qs = [
    "What legal requirements are mentioned?",
    "Are there any penalties discussed?",
    "What compliance obligations exist?",
]
for q in sample_qs:
    console.print(f"[dim]  • {q}[/dim]")

console.print("\n[bold cyan]Enter your question below:[/bold cyan]")

# Note: In SageMaker, you'll need to run this cell multiple times
# Each time, update the 'user_question' variable below

user_question = "What are the main legal requirements?"  # ← Change this

console.print(f"\n[bold cyan]❓ Your Question:[/bold cyan] {user_question}\n")

answer, citations = query_rag(user_question)

console.print(Panel(
    answer,
    title="[bold green]🤖 CLaRa Response[/bold green]",
    border_style="green"
))

if citations:
    console.print("\n[bold cyan]📚 Sources:[/bold cyan]")
    for i, cite in enumerate(citations, 1):
        console.print(
            f"  {i}. {cite['document']} "
            f"(Pages: {cite['pages']}, "
            f"Relevance Score: {cite['score']:.2f})"
        )

╭──────────────────────────────────────────╮
│ 🤖 CLaRa Interactive Chat                │
│ Ask questions about your legal documents │
╰──────────────────────────────────────────╯

Try these questions:

  • What legal requirements are mentioned?

  • Are there any penalties discussed?

  • What compliance obligations exist?

...

╭─────────────────────────────────────────────── 🤖 CLaRa Response ───────────────────────────────────────────────╮
│ According to the Mississippi Rules of Appellate Procedure, Rule 46(b), all documents, except for prefiled       │
│ testimony, must be signed by an attorney of record or the party itself, with contact information provided       │
│ (Documents 2 and 3). Attorneys must certify they have read the document and believe it is valid. Foreign        │
│ attorneys, in addition, must certify their admission in accordance with Rule 46(b) (Document 1). Unsigned or    │
│ intentionally false documents may be struck. Maps and plans, when required, must be filed in original scale     │
│ with the Executive Secretary of the Commission, providing two copies (Document 1).                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📚 Sources:

1. tmp221d6xvh.pdf (Pages: [29, 30], Relevance Score: 0.72)

2. 00000047c.pdf (Pages: [29], Relevance Score: 0.71)

3. tmp221d6xvh.pdf (Pages: [29], Relevance Score: 0.71)